# Categorical Co-occurrence of Lightning and Overshooting Tops (3D Dilation Matching)

Purpose: Classify lightning and overshooting-top (OT) detections into three categories :
"LS only" (lightning with no co-occurring OT), "LS+OT" (co-occurring), and "OT only" (OT
with no co-occurring lightning) — separately for land and ocean, using a 3D (time, lat,
lon) morphological dilation of the gridded lightning mask to allow for spatiotemporal
tolerance when matching against OT point detections.

Input:
- Gridded lightning mask (NetCDF): lightning_mask.nc
- Land/ocean lightning point data (csv): land_lightning.csv,
  ocean_lightning.csv
- Land/ocean OT point data (feather): whole_Australia_LAND_OT_2016_2024.feather,
  whole_Australia_OCEAN_OT_2016_2024.feather
- Region of interest: Northern Australia (lat: -20 to -10, lon: 120 to 148)
- Date range: 2016-01-01 to 2024-12-31

Processing:
1. Load the gridded lightning mask and apply 3D binary dilation (3x3x3 neighborhood in
   time/lat/lon) to create a spatiotemporally tolerant "nearby lightning" mask and its
   inverse ("no lightning nearby").
2. Load land/ocean lightning and OT point dataframes; round coordinates and restrict to
   the region of interest.
3. For each OT point, sample the dilated lightning mask at the nearest grid cell/time
   step (nearest-neighbor matching against the gridded mask) to classify it as
   OT-with-nearby-lightning ("LS+OT") or OT-only.
4. For lightning points, identify those with no exact matching OT point (direct
   point-to-point comparison, since both are dataframes) to obtain "LS only" points.
5. Validate internal consistency: total lightning = LS-only + LS+OT (This won't be true, as
   I dilated it to identify the OT with nearby lightning), and total OT = OT-only + LS+OT
   (this must be true), for both land and ocean.
7. Summarize counts per category (LS only / LS+OT / OT only) by land and ocean.

Output:
- Per-category CSVs (land/ocean, LS-only/LS+OT/OT-only) saved to:
  /g/data/if69/tp4064/Project_1/categorical_position/Dilation/
- Console summary table of category counts by land/ocean.

In [6]:
import numpy as np
import xarray as xr
# from scipy.ndimage import label, generate_binary_structure
from scipy.ndimage import binary_dilation
import pandas as pd
import pyarrow


In [ ]:
def dilate_3d(da, size=(3, 3, 3)):
    """
    Morphological dilation on boolean DataArray with dims (time, lat, lon),
    applied across all three dimensions.

    Parameters
    ----------
    da : xarray.DataArray (bool)
        Input with dims (time, lat, lon).
    size : tuple of 3 ints
        Neighborhood size along (time, lat, lon).
        Example: (3, 3, 3) = 3 steps in time, 3 in lat, 3 in lon.
        Example: (1, 3, 3) = only spatial dilation (no time).
    """
    # Structuring element: full cube of True
    footprint = np.ones(size, dtype=bool)

    return xr.apply_ufunc(
        binary_dilation,
        da,
        kwargs={"structure": footprint},
        input_core_dims=[["time", "lat", "lon"]],
        output_core_dims=[["time", "lat", "lon"]],
        dask="parallelized",
        output_dtypes=[bool],
    )

#  dilate by 1 step in time, 1 in lat, 1 in lon
# dilated_ls = dilate_time_latlon_bool(lightning_data_select["lightning_count"], size=(3, 3, 3))



def select_point_based_data(data, variable):

    """
    Filter Data based on input given, specifically design for Land/Ocean Region for OT.
    This function filters the dataset to a specific lat/lon box and removes data from 2015.
    and select the nearest grid points based on points given
    """
    # data must be dataset
    #convert to time to datetime
    variable['time'] = pd.to_datetime(variable['time'])
    
    #
    variable['lat'] = variable['lat'].round(3)
    variable['lon'] = variable['lon'].round(3)

    # data to be sorted by time
    data = data.sortby('time')
    
    #Create DataArrays for selection
    da_lon_ot = xr.DataArray(variable['lon'], dims=['ot'])
    da_lat_ot = xr.DataArray(variable['lat'], dims=['ot'])
    da_time_ot = xr.DataArray(variable['time'], dims=['ot'])

    # Select nearest data points
    data_select = data.sel(lat = da_lat_ot, lon = da_lon_ot, time = da_time_ot, method = 'nearest', )

    return data_select


def ls_only(lightning_df, ot_df):
    """
    finding lightning only points
    """
    OT_points = ot_df[['lat','lon','time']].apply(tuple, axis=1)
    # OT_points
    lightning_points = lightning_df[['lat','lon','time']].apply(tuple, axis=1)
    # lightning_points
    mask = ~lightning_points.isin(OT_points)
    ls_no_OT = lightning_points[mask]
    ls_no_OT = pd.DataFrame(ls_no_OT.tolist(), columns=['lat', 'lon', 'time'])
    

    return ls_no_OT


In [7]:
#select the regions ad time
lat_max = -10 #start for upto -10
lat_min = -20 #stop for upto -20
lon_min = 120 #start for upto 120
lon_max = 148 #stop for upto 148

start_date = '2016-01-01' ## YYYY-MM-DD
end_date = '2024-12-31'

In [17]:
## Load and Process Lightning Data for Lightning mask
# # rounding the lat and lon is essential as they affects the selection of points

# Lightning_mask can be simply generated by taking all file together and using condition of  Lightning _count > 1

lightning_mask = xr.open_dataset('/g/data/if69/tp4064/lightning_mask_data/lightning_mask.nc') 
# no_lightning_mask = ~lightning_mask.astype(bool)

dilate_3d_ls_mask = dilate_3d(lightning_mask["lightning_count"], size=(3, 3, 3))
dilate_3d_ls_mask = dilate_3d_ls_mask.to_dataset(name='lightning')

# no_dilate_3d_ls_mask = ~dilate_3d_ls_mask
no_dilate_3d_ls_mask = (~dilate_3d_ls_mask.astype(bool)).rename({'lightning': 'no_lightning'})

land_lightning = pd.read_csv("/g/data/if69/tp4064/Project_1/Lightning/land_lightning.csv", 
                             usecols = ['lat','lon','time'],
                             parse_dates = ['time'])

land_lightning['lat'] = land_lightning['lat'].round(3)
land_lightning['lon'] = land_lightning['lon'].round(3)

ocean_lightning = pd.read_csv("/g/data/if69/tp4064/Project_1/Lightning/ocean_lightning.csv", 
                              usecols = ['lat','lon','time'],
                              parse_dates = ['time'])

ocean_lightning['lat'] = ocean_lightning['lat'].round(3)
ocean_lightning['lon'] = ocean_lightning['lon'].round(3)


land_OT = pd.read_feather(
    "/g/data/if69/tp4064/Project_1/OT/dataframe_LAND_OT/whole_Australia_LAND_OT_2016_2024.feather",
    columns=['lat', 'lon', 'UTC_time']  # usecols → columns in feather
)

# parse_dates doesn't exist in read_feather
# feather preserves dtypes automatically, so just convert after if needed
land_OT['UTC_time'] = pd.to_datetime(land_OT['UTC_time'])


land_OT.rename(columns={'UTC_time': 'time'}, inplace=True)
land_OT = land_OT[((land_OT['lat'] <= -10)&(land_OT['lat'] >= -20)) & ((land_OT['lon'] <= 148)&(land_OT['lon'] >= 120))]

land_OT['lat'] = land_OT['lat'].round(3)
land_OT['lon'] = land_OT['lon'].round(3)

ocean_OT = pd.read_feather("/g/data/if69/tp4064/Project_1/OT/dataframe_OCEAN_OT/whole_Australia_OCEAN_OT_2016_2024.feather",
                     columns = ['lat','lon','UTC_time'])
                     #         parse_dates = ['UTC_time'])

ocean_OT['UTC_time'] = pd.to_datetime(ocean_OT['UTC_time'])

ocean_OT.rename(columns={'UTC_time': 'time'}, inplace=True)
ocean_OT = ocean_OT[((ocean_OT['lat'] <= -10)&(ocean_OT['lat'] >= -20)) & ((ocean_OT['lon'] <= 148)&(ocean_OT['lon'] >= 120))]


ocean_OT['lat'] = ocean_OT['lat'].round(3)
ocean_OT['lon'] = ocean_OT['lon'].round(3)

In [18]:
%%time
ot_only_land_dilate_3d = select_point_based_data(no_dilate_3d_ls_mask, land_OT).to_dataframe()
ot_only_land_dilate_3d=ot_only_land_dilate_3d[ot_only_land_dilate_3d['no_lightning'] == True]

ot_only_ocean_dilate_3d = select_point_based_data(no_dilate_3d_ls_mask, ocean_OT).to_dataframe()
ot_only_ocean_dilate_3d=ot_only_ocean_dilate_3d[ot_only_ocean_dilate_3d['no_lightning'] == True]

ls_ot_land_dilate_3d = (select_point_based_data(dilate_3d_ls_mask, land_OT)).to_dataframe()
ls_ot_land_dilate_3d=ls_ot_land_dilate_3d[ls_ot_land_dilate_3d['lightning'] == True]

ls_ot_ocean_dilate_3d = (select_point_based_data(dilate_3d_ls_mask, ocean_OT)).to_dataframe()
ls_ot_ocean_dilate_3d=ls_ot_ocean_dilate_3d[ls_ot_ocean_dilate_3d['lightning'] == True]

ls_only_land = ls_only(land_lightning, land_OT)
ls_only_ocean = ls_only(ocean_lightning, ocean_OT)

CPU times: user 1min 19s, sys: 6.74 s, total: 1min 26s
Wall time: 1min 26s


In [19]:
# just checking 
print(
    "Land Lightning:",
    len(land_lightning),
    "=",
    len(ls_only_land),
    "+",
    len(ls_ot_land_dilate_3d),
    "->",
    len(land_lightning) == len(ls_only_land) + len(ls_ot_land_dilate_3d)
)

print(
    "Ocean Lightning:",
    len(ocean_lightning),
    "=",
    len(ls_only_ocean),
    "+",
    len(ls_ot_ocean_dilate_3d),
    "->",
    len(ocean_lightning) == len(ls_only_ocean) + len(ls_ot_ocean_dilate_3d)
)

print(
    "Land OT:",
    len(land_OT),
    "=",
    len(ot_only_land_dilate_3d),
    "+",
    len(ls_ot_land_dilate_3d),
    "->",
    len(land_OT) == len(ot_only_land_dilate_3d) + len(ls_ot_land_dilate_3d)
)

print(
    "Ocean OT:",
    len(ocean_OT),
    "=",
    len(ot_only_ocean_dilate_3d),
    "+",
    len(ls_ot_ocean_dilate_3d),
    "->",
    len(ocean_OT) == len(ot_only_ocean_dilate_3d) + len(ls_ot_ocean_dilate_3d)
)

Land Lightning: 5900737 = 5512945 + 510625 -> False
Ocean Lightning: 2519333 = 2185344 + 576420 -> False
Land OT: 547339 = 36714 + 510625 -> True
Ocean OT: 753018 = 176598 + 576420 -> True


In [28]:
# this is for Table in the paper
summary = pd.DataFrame([
    {
        'Category'       : 'LS only',
        'Land'          : len(ls_only_land),
        'Ocean'        : len(ls_only_ocean),
        
    },
    {
        'Category'       : 'LS+OT',
        'Land'          : len(ls_ot_land_dilate_3d),
        'Ocean'        : len(ls_ot_ocean_dilate_3d),
        
    },
    {
        'Category'       : 'OT only',
        'Land'          : len(ot_only_land_dilate_3d),
        'Ocean'        : len(ot_only_ocean_dilate_3d),
        
    }
    
]).set_index('Category')

print(summary)

             Land    Ocean
Category                  
LS only   5512945  2185344
LS+OT      510625   576420
OT only     36714   176598


In [24]:
# saving files
ot_only_ocean_dilate_3d.to_csv("/g/data/if69/tp4064/Project_1/categorical_position/Dilation/ot_only_ocean_3d.csv")
ot_only_land_dilate_3d.to_csv("/g/data/if69/tp4064/Project_1/categorical_position/Dilation/ot_only_land_3d.csv")
ls_ot_land_dilate_3d.to_csv("/g/data/if69/tp4064/Project_1/categorical_position/Dilation/ls_ot_land_3d.csv")
ls_ot_ocean_dilate_3d.to_csv("/g/data/if69/tp4064/Project_1/categorical_position/Dilation/ls_ot_ocean_3d.csv")
ls_only_land.to_csv("/g/data/if69/tp4064/Project_1/categorical_position/Dilation/ls_only_land_3d.csv")
ls_only_ocean.to_csv("/g/data/if69/tp4064/Project_1/categorical_position/Dilation/ls_only_ocean_3d.csv")